# Build Cross-Rotation Splits

Generate the official 8-fold cross-rotation split CSVs from the processed manifest.

In [ ]:
from __future__ import annotations

from meatlens_pork_pipeline.notebook_progress import (
    advance_notebook_cell_progress,
    finish_notebook_cell_progress,
    iter_notebook_progress,
    start_notebook_cell_progress,
)
NB_03_BUILD_CROSS_ROTATION_SPLITS_CELL_PROGRESS_1 = start_notebook_cell_progress('03_build_cross_rotation_splits.ipynb', 'Load shared setup', total_steps=1)

import json
import re
from pathlib import Path

import pandas as pd

shared_notebook = json.loads(Path('00_shared_setup.ipynb').read_text(encoding='utf-8'))
shared_code = '\n\n'.join(
    ''.join(cell.get('source', []))
    for cell in shared_notebook['cells']
    if cell.get('cell_type') == 'code'
)
exec(shared_code, globals())

finish_notebook_cell_progress(NB_03_BUILD_CROSS_ROTATION_SPLITS_CELL_PROGRESS_1)


In [ ]:
from meatlens_pork_pipeline.notebook_progress import (
    advance_notebook_cell_progress,
    finish_notebook_cell_progress,
    iter_notebook_progress,
    start_notebook_cell_progress,
)
NB_03_BUILD_CROSS_ROTATION_SPLITS_CELL_PROGRESS_2 = start_notebook_cell_progress('03_build_cross_rotation_splits.ipynb', 'Define split helpers', total_steps=1)

def sample_sort_key(sample_id: str) -> tuple[int, str]:
    match = re.search(r'(\d+)$', str(sample_id))
    if match:
        return int(match.group(1)), str(sample_id)
    return 10**9, str(sample_id)


def require_official_sample_ids(manifest_df: pd.DataFrame, expected_count: int = 8) -> pd.DataFrame:
    if 'sample_id' not in manifest_df.columns:
        raise ValueError('Processed manifest must contain a sample_id column.')
    df = manifest_df.copy()
    df['sample_id'] = df['sample_id'].astype(str).str.strip()
    if not df['sample_id'].all():
        raise ValueError('Every processed row must have a non-empty sample_id.')
    unique_sample_ids = sorted(df['sample_id'].unique(), key=sample_sort_key)
    if len(unique_sample_ids) != expected_count:
        raise ValueError(
            f'Official cross-rotation requires exactly {expected_count} unique sample_ids. Found {len(unique_sample_ids)}.'
        )
    return df


def build_official_cross_rotation_splits(
    processed_df: pd.DataFrame,
    output_root: Path,
) -> tuple[dict[str, Path], pd.DataFrame, pd.DataFrame]:
    df = require_official_sample_ids(processed_df)
    ensure_dir(output_root)

    unique_sample_ids = sorted(df['sample_id'].unique(), key=sample_sort_key)
    summary_rows: list[dict[str, object]] = []
    leakage_rows: list[dict[str, object]] = []
    written_paths: dict[str, Path] = {}

    all_sampled_images_path = output_root / 'all_sampled_images.csv'
    df.to_csv(all_sampled_images_path, index=False)
    written_paths['all_sampled_images'] = all_sampled_images_path

    for fold_index, test_sample_id in enumerate(
        iter_notebook_progress(
            unique_sample_ids,
            '03_build_cross_rotation_splits.ipynb | build official folds',
            total=len(unique_sample_ids),
            unit='fold',
            leave=True,
        ),
        start=1,
    ):
        val_sample_id = unique_sample_ids[fold_index % len(unique_sample_ids)]
        train_sample_ids = [sample_id for sample_id in unique_sample_ids if sample_id not in {test_sample_id, val_sample_id}]

        fold_name = f'fold{fold_index}'
        train_df = df[df['sample_id'].isin(train_sample_ids)].copy()
        val_df = df[df['sample_id'] == val_sample_id].copy()
        test_df = df[df['sample_id'] == test_sample_id].copy()

        for split_name, split_df in (('train', train_df), ('val', val_df), ('test', test_df)):
            split_df['fold'] = fold_name
            split_df['split'] = split_name
            split_df['split_type'] = 'cross_rotation'
            output_path = output_root / f'{fold_name}_{split_name}.csv'
            split_df.to_csv(output_path, index=False)
            written_paths[f'{fold_name}_{split_name}'] = output_path

        summary_rows.append(
            {
                'fold': fold_name,
                'train_samples': '|'.join(train_sample_ids),
                'val_sample': val_sample_id,
                'test_sample': test_sample_id,
                'train_count': len(train_df),
                'val_count': len(val_df),
                'test_count': len(test_df),
            }
        )

        leakage_rows.append(
            {
                'fold': fold_name,
                'train_val_overlap': bool(set(train_df['sample_id']) & set(val_df['sample_id'])),
                'train_test_overlap': bool(set(train_df['sample_id']) & set(test_df['sample_id'])),
                'val_test_overlap': bool(set(val_df['sample_id']) & set(test_df['sample_id'])),
            }
        )

    summary_df = pd.DataFrame(summary_rows)
    leakage_df = pd.DataFrame(leakage_rows)
    summary_path = output_root / 'cross_rotation_summary.csv'
    leakage_path = output_root / 'cross_rotation_leakage_check.csv'
    summary_df.to_csv(summary_path, index=False)
    leakage_df.to_csv(leakage_path, index=False)
    written_paths['summary'] = summary_path
    written_paths['leakage'] = leakage_path
    return written_paths, summary_df, leakage_df

finish_notebook_cell_progress(NB_03_BUILD_CROSS_ROTATION_SPLITS_CELL_PROGRESS_2)


In [ ]:
from meatlens_pork_pipeline.notebook_progress import (
    advance_notebook_cell_progress,
    finish_notebook_cell_progress,
    iter_notebook_progress,
    start_notebook_cell_progress,
)
NB_03_BUILD_CROSS_ROTATION_SPLITS_CELL_PROGRESS_3 = start_notebook_cell_progress('03_build_cross_rotation_splits.ipynb', 'Build official folds', total_steps=1)

PROCESSED_MANIFEST_PATH = Path(str(override('PROCESSED_MANIFEST_PATH', GENERATED_SPLITS_ROOT / 'processed_manifest.csv')))
CROSS_ROTATION_OUTPUT_ROOT = Path(str(override('CROSS_ROTATION_OUTPUT_ROOT', GENERATED_SPLITS_ROOT)))

processed_df = pd.read_csv(PROCESSED_MANIFEST_PATH, dtype=str).fillna('')
written_paths, summary_df, leakage_df = build_official_cross_rotation_splits(
    processed_df,
    output_root=CROSS_ROTATION_OUTPUT_ROOT,
)

print(f'Generated fold files: {len(written_paths)}')
print(summary_df[['fold', 'val_sample', 'test_sample']].to_string(index=False))

finish_notebook_cell_progress(NB_03_BUILD_CROSS_ROTATION_SPLITS_CELL_PROGRESS_3)
